# Supplementary Figure — PCA validation (condensed overlay)

This revised supplementary notebook condenses the PCA validation figure.

## New layout
All three conditions are now shown on **one shared plot**.

- **Condition** is encoded by color:
  - intact = purple
  - word-scrambled = green
  - rest = black
- **PCA space** is encoded by line style:
  - solid = reconstruction
  - dashed = projected

Condition-specific raw HTFA references are shown as color-matched dotted
horizontal lines, and chance is shown as a gray dotted line.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

CONDITIONS = ["intact", "word", "rest"]
CONDITION_LABELS = {
    "intact": "Intact narrative",
    "word": "Word-scrambled",
    "rest": "Rest",
}
CONDITION_COLORS = {
    "intact": "#7B3294",
    "word": "#2CA25F",
    "rest": "#222222",
}

T = 300
V = 700

PCA_DIR = Path("pca_aa_equivalence_outputs")
RAW_CSV = PCA_DIR / "raw_htfa_summary.csv"
PCA_CSV = PCA_DIR / "pca_summary.csv"

SPATIAL_FULL_CSV = Path(
    "msaa_condrank_decoding_outputs_spatial_across/full_summary.csv"
)
TEMPORAL_FULL_CSV = Path(
    "msaa_condrank_decoding_outputs_temporal_across/full_summary.csv"
)

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_PCA_validation_condensed.png"
FIG_PDF = OUTPUT_DIR / "Supplement_PCA_validation_condensed.pdf"
SOURCE_CSV = OUTPUT_DIR / "Supplement_PCA_validation_source_data.csv"

DPI = 300

for p in [RAW_CSV, PCA_CSV, SPATIAL_FULL_CSV, TEMPORAL_FULL_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required input: {p}\n"
            "Run the corresponding upstream analysis notebook first."
        )

In [ ]:
raw = pd.read_csv(RAW_CSV)
pca = pd.read_csv(PCA_CSV)

spatial = pd.read_csv(SPATIAL_FULL_CSV)
spatial["analysis_type"] = "spatial"
if "component_ratio" not in spatial:
    spatial["component_ratio"] = spatial["K"].astype(float) / V

temporal = pd.read_csv(TEMPORAL_FULL_CSV)
temporal["analysis_type"] = "temporal"
if "component_ratio" not in temporal:
    temporal["component_ratio"] = temporal["K"].astype(float) / T

aa = pd.concat([spatial, temporal], ignore_index=True)

if "component_ratio" not in pca:
    pca["component_ratio"] = pca["K"].astype(float) / V

print("Raw rows:", len(raw))
print("PCA rows:", len(pca))
print("AA rows:", len(aa))

In [ ]:
# Export a single combined source-data table for the paper figure.
raw_export = raw.copy()
raw_export["source"] = "raw_htfa"

pca_export = pca.copy()
pca_export["source"] = "pca"

aa_export = aa.copy()
aa_export["source"] = "msaa"

all_cols = sorted(
    set(raw_export.columns)
    | set(pca_export.columns)
    | set(aa_export.columns)
)

pd.concat(
    [
        raw_export.reindex(columns=all_cols),
        pca_export.reindex(columns=all_cols),
        aa_export.reindex(columns=all_cols),
    ],
    ignore_index=True,
).to_csv(SOURCE_CSV, index=False)

print("Saved source data:", SOURCE_CSV)

In [ ]:

from matplotlib.lines import Line2D

# ============================================================
# SUPPLEMENTARY PCA VALIDATION — CONDENSED OVERLAY
# ============================================================

COND_COLORS = {
    "intact": "#7B3294",
    "word":   "#2CA25F",
    "rest":   "#222222",
}

COND_LABELS = {
    "intact": "Intact narrative",
    "word":   "Word-scrambled",
    "rest":   "Rest",
}

fig, ax = plt.subplots(figsize=(8.8, 6.2))

for condition in CONDITIONS:
    color = COND_COLORS[condition]

    raw_row = raw[raw["condition"].astype(str) == condition]
    raw_value = float(raw_row.iloc[0]["mean"]) if len(raw_row) else np.nan

    p_proj = (
        pca[
            (pca["condition"].astype(str) == condition)
            & (pca["decode_space"].astype(str) == "projected")
        ]
        .sort_values("component_ratio")
    )

    p_rec = (
        pca[
            (pca["condition"].astype(str) == condition)
            & (pca["decode_space"].astype(str) == "reconstruction")
        ]
        .sort_values("component_ratio")
    )

    if len(p_proj):
        ax.errorbar(
            p_proj["component_ratio"],
            p_proj["mean"],
            yerr=(p_proj["err"] if "err" in p_proj.columns else None),
            color=color,
            linestyle="--",
            marker="o",
            markersize=4.0,
            linewidth=1.9,
            capsize=2,
            alpha=0.85,
        )

    if len(p_rec):
        ax.errorbar(
            p_rec["component_ratio"],
            p_rec["mean"],
            yerr=(p_rec["err"] if "err" in p_rec.columns else None),
            color=color,
            linestyle="-",
            marker="o",
            markersize=4.2,
            linewidth=2.1,
            capsize=2,
            alpha=0.98,
        )

    if np.isfinite(raw_value):
        ax.axhline(
            raw_value,
            color=color,
            linestyle=":",
            linewidth=1.8,
            alpha=0.90,
        )

chance = 1.0 / T
ax.axhline(chance, color="0.60", linestyle=":", linewidth=1.2)

ax.set_title("PCA validation: projected vs reconstructed decoding", fontsize=20, fontweight="bold", pad=14)
ax.set_xlabel("Normalized representational ratio", fontsize=13)
ax.set_ylabel("Timepoint-identification accuracy", fontsize=13)
ax.set_xlim(0, 1.02)
ax.tick_params(axis="both", labelsize=11)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

cond_handles = [
    Line2D([0], [0], color=COND_COLORS[c], lw=2.5, label=COND_LABELS[c])
    for c in ["intact", "word", "rest"]
]
style_handles = [
    Line2D([0], [0], color="0.2", lw=2.0, linestyle="-", marker="o", markersize=5, label="PCA reconstruction"),
    Line2D([0], [0], color="0.2", lw=1.9, linestyle="--", marker="o", markersize=5, label="PCA projected"),
    Line2D([0], [0], color="0.2", lw=1.8, linestyle=":", label="Raw HTFA"),
    Line2D([0], [0], color="0.60", lw=1.2, linestyle=":", label="Chance"),
]

leg1 = ax.legend(
    handles=cond_handles,
    title="Condition",
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.28, -0.34),
    ncol=1,
    fontsize=10,
    title_fontsize=10,
)
ax.add_artist(leg1)

ax.legend(
    handles=style_handles,
    title="Curve type",
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.76, -0.34),
    ncol=1,
    fontsize=10,
    title_fontsize=10,
)

fig.subplots_adjust(left=0.12, right=0.98, top=0.90, bottom=0.32)

fig.savefig(FIG_PNG, dpi=DPI, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)


### Supplementary interpretation

This condensed supplementary figure places all three conditions on the same
axis so the PCA validation can be read quickly. Color indicates condition,
while line style distinguishes PCA reconstruction from PCA projected decoding.
